# Analysis
It is time to analyse some data. We here show how to set up an Analysis object and use it to first fit an artificial vanadium measurements, and next an artificial measurement of a model with diffusion and some elastic scattering.

In the near future, it will be possible to fit the width and area of the Lorentzian to the diffusion model, as well as fitting the diffusion model directly to the data.

In [ ]:
import matplotlib.pyplot as plt

from easydynamics.analysis.analysis1d import Analysis1d
from easydynamics.experiment import Experiment
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import DeltaFunction
from easydynamics.sample_model import Lorentzian
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model import Polynomial
from easydynamics.sample_model.background_model import BackgroundModel
from easydynamics.sample_model.resolution_model import ResolutionModel
from easydynamics.sample_model.sample_model import SampleModel
from easydynamics.sample_model.instrument_model import InstrumentModel
from easydynamics.analysis.analysis import Analysis
from copy import copy
%matplotlib widget

In [ ]:
vanadium_experiment = Experiment('Vanadium')
vanadium_experiment.load_hdf5(filename='vanadium_data_example.h5')

In [ ]:
# Example of Analysis with a simple sample model and instrument model
delta_function = DeltaFunction(display_name='DeltaFunction', area=1)
sample_model = SampleModel(
    components=delta_function,
)

res_gauss = Gaussian(width=0.1)
res_gauss.area.fixed=True
resolution_model = ResolutionModel(components=res_gauss)


background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

instrument_model = InstrumentModel(
    resolution_model=resolution_model,
    background_model=background_model,
)

vanadium_analysis = Analysis(
    display_name='Vanadium Full Analysis',
    experiment=vanadium_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

fit_result_independent_single_Q = vanadium_analysis.fit(fit_method="independent", Q_index=5)
vanadium_analysis.plot_data_and_model(Q_index=5)

In [ ]:
fit_result_independent_all_Q = vanadium_analysis.fit(fit_method="independent")
vanadium_analysis.plot_data_and_model()

In [ ]:
fit_result_simultaneous = vanadium_analysis.fit(fit_method="simultaneous")
fit_result_simultaneous
vanadium_analysis.plot_data_and_model()

In [ ]:
# Inspect the Parameters as a scipp Dataset
vanadium_analysis.parameters_to_dataset()


In [ ]:
# Plot some of fitted parameters as a function of Q
vanadium_analysis.plot_parameters(names=["DeltaFunction area"])


In [ ]:
vanadium_analysis.plot_parameters(names=["Gaussian width"])

In [ ]:
# Set up the diffusion analysis with the same resolution model as the
# vanadium analysis
diffusion_experiment = Experiment('Diffusion')
diffusion_experiment.load_hdf5(filename='diffusion_data_example.h5')

In [ ]:
# We set up the model first.
delta_function = DeltaFunction(display_name='DeltaFunction', area=0.2)
lorentzian = Lorentzian(display_name='Lorentzian', area=0.5, width=0.3)
component_collection=ComponentCollection(
    components=[delta_function, lorentzian],
)
sample_model = SampleModel(
    components=component_collection,
)

background_model = BackgroundModel(components=Polynomial(coefficients=[0.001]))

instrument_model = InstrumentModel(
    background_model=background_model,
)

diffusion_analysis = Analysis(
    display_name='Diffusion Full Analysis',
    experiment=diffusion_experiment,
    sample_model=sample_model,
    instrument_model=instrument_model,
)

# We need to hack in the resolution model from the vanadium analysis,
# since the setters and getters overwrite the model. This will be fixed
# asap.
diffusion_analysis.instrument_model._resolution_model = vanadium_analysis.instrument_model.resolution_model
diffusion_analysis.instrument_model.resolution_model.fix_all_parameters()
diffusion_analysis.plot_parameters(names=["Gaussian width"])


In [ ]:
# Let us see how good the starting parameters are
diffusion_analysis.plot_data_and_model()

In [ ]:
# Now we fit the data and plot the result. Looks good!
diffusion_analysis.fit(fit_method="independent")
diffusion_analysis.plot_data_and_model()

In [ ]:
# Let us look at the most interesting fit parameters
diffusion_analysis.plot_parameters(names=["Lorentzian width", "Lorentzian area"])

In [ ]:
# It will be possible to fit this to a DiffusionModel, but that will
# come later.